# PipelineWatch-NG — Module 3: ML Anomaly Detection (Fixed)
### XGBoost + Isolation Forest + SHAP

**Fix applied:** matplotlib Agg backend — all plots save to outputs/ without crashing.

---

## Cell 1 — Imports

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from IPython.display import Image, display
import os, json, gc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from datetime import datetime
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
import xgboost as xgb
import joblib

CACHE_DIR  = "../data/cached"
OUTPUT_DIR = "../outputs"
MODEL_DIR  = "../data/models"
os.makedirs(MODEL_DIR, exist_ok=True)

print("Imports OK. XGBoost: " + xgb.__version__)
print("matplotlib backend: " + matplotlib.get_backend())


Imports OK. XGBoost: 3.2.0
matplotlib backend: Agg


## Cell 2 — Load feature table

In [2]:
df = pd.read_csv(os.path.join(CACHE_DIR, "m2_feature_table.csv"))
print("Feature table: " + str(df.shape[0]) + " rows x " + str(df.shape[1]) + " cols")
print("Columns: " + str(df.columns.tolist()))
print()
print(df.describe().round(3))


Feature table: 270 rows x 13 cols
Columns: ['NDMI', 'NDVI', 'NDVI_change', 'NDWI', 'SAR_change_dB', 'VH', 'VV', 'chronic_spill_mask', 'dark_spot_magnitude', 'dark_spot_mask', 'new_spill_mask', 'lon', 'lat']

          NDMI     NDVI  NDVI_change     NDWI  SAR_change_dB       VH  \
count  270.000  270.000      270.000  270.000        270.000  270.000   
mean     0.089    0.493        0.033   -0.469         -0.023  -14.543   
std      0.082    0.172        0.102    0.149          0.417    1.258   
min     -0.162   -0.242       -0.321   -0.681         -1.487  -23.754   
25%      0.041    0.396       -0.036   -0.570         -0.253  -15.007   
50%      0.088    0.521        0.041   -0.496         -0.020  -14.552   
75%      0.143    0.611        0.108   -0.392          0.166  -13.984   
max      0.309    0.768        0.272    0.253          2.831   -9.252   

            VV  chronic_spill_mask  dark_spot_magnitude  dark_spot_mask  \
count  270.000             270.000              270.000    

## Cell 3 — Feature selection and weak labelling

In [3]:
FEATURE_COLS = ["VV","VH","dark_spot_mask","dark_spot_magnitude",
                "SAR_change_dB","new_spill_mask","chronic_spill_mask"]
for col in ["NDVI","NDWI","NDMI","NDVI_change"]:
    if col in df.columns: FEATURE_COLS.append(col)

df_clean = df[FEATURE_COLS + ["lat","lon"]].dropna().copy()
print("Features: " + str(FEATURE_COLS))
print("Clean rows: " + str(len(df_clean)))

def assign_risk_label(row):
    sar_dark   = row["dark_spot_mask"] == 1
    sar_change = row["SAR_change_dB"] < -0.5
    chronic    = row["chronic_spill_mask"] == 1
    dark_mag   = row["dark_spot_magnitude"] > 1.0
    vv_low     = row["VV"] < -10.0
    ndvi_drop  = row.get("NDVI_change", 0) < -0.10 if "NDVI_change" in row.index else False
    if (sar_dark or chronic) and (dark_mag or ndvi_drop): return 2
    elif sar_change or vv_low or ndvi_drop or dark_mag:   return 1
    else:                                                  return 0

df_clean["risk_label"] = df_clean.apply(assign_risk_label, axis=1)
label_counts = df_clean["risk_label"].value_counts().sort_index()
print()
print("Risk label distribution:")
for label, count in label_counts.items():
    name = {0:"LOW",1:"MEDIUM",2:"HIGH"}[label]
    print("  " + name + ": " + str(count) + " (" + str(round(100*count/len(df_clean),1)) + "%)")


Features: ['VV', 'VH', 'dark_spot_mask', 'dark_spot_magnitude', 'SAR_change_dB', 'new_spill_mask', 'chronic_spill_mask', 'NDVI', 'NDWI', 'NDMI', 'NDVI_change']
Clean rows: 270

Risk label distribution:
  LOW: 214 (79.3%)
  MEDIUM: 53 (19.6%)
  HIGH: 3 (1.1%)


## Cell 4 — Train XGBoost

In [4]:
X = df_clean[FEATURE_COLS].values
y = df_clean["risk_label"].values
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)
print("Train: " + str(len(X_train)) + "  Test: " + str(len(X_test)))

xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=4, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="mlogloss", random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred    = xgb_model.predict(X_test)
cv_scores = cross_val_score(xgb_model, X_scaled, y, cv=5, scoring="accuracy")

print(classification_report(y_test, y_pred,
      target_names=["LOW","MEDIUM","HIGH"], zero_division=0))
print("5-fold CV accuracy: " + str(round(cv_scores.mean(),3))
      + " +/- " + str(round(cv_scores.std(),3)))


Train: 216  Test: 54
              precision    recall  f1-score   support

         LOW       0.95      1.00      0.98        42
      MEDIUM       1.00      0.91      0.95        11
        HIGH       0.00      0.00      0.00         1

    accuracy                           0.96        54
   macro avg       0.65      0.64      0.64        54
weighted avg       0.95      0.96      0.95        54

5-fold CV accuracy: 0.981 +/- 0.017


## Cell 5 — Feature importance chart

In [6]:
import plotly.graph_objects as go

importances = xgb_model.feature_importances_
feat_imp = pd.DataFrame({
    'feature':    FEATURE_COLS,
    'importance': importances
}).sort_values('importance', ascending=True)

print('Feature importances:')
print(feat_imp.sort_values('importance', ascending=False).to_string(index=False))

colors = ['#E24B4A' if i >= len(feat_imp)-3 else '#378ADD'
          for i in range(len(feat_imp))]

fig = go.Figure(go.Bar(
    x=feat_imp['importance'],
    y=feat_imp['feature'],
    orientation='h',
    marker_color=colors,
    opacity=0.85
))

fig.add_vline(x=feat_imp['importance'].mean(),
              line_dash='dash', line_color='#854F0B',
              annotation_text='Mean')

fig.update_layout(
    title='PipelineWatch-NG - Feature Importance (XGBoost gain)',
    xaxis_title='Importance score',
    height=420
)

fig.write_html(os.path.join(OUTPUT_DIR, 'm3_feature_importance.html'))
print('Saved: outputs/m3_feature_importance.html')
from IPython.display import IFrame
display(IFrame(src='../outputs/m3_feature_importance.html', width='100%', height=450))

Feature importances:
            feature  importance
      SAR_change_dB    0.282266
        NDVI_change    0.239312
                 VV    0.093400
                 VH    0.087711
               NDVI    0.081753
               NDMI    0.077943
               NDWI    0.077565
dark_spot_magnitude    0.060050
 chronic_spill_mask    0.000000
     dark_spot_mask    0.000000
     new_spill_mask    0.000000
Saved: outputs/m3_feature_importance.html


## Cell 6 — Isolation Forest

In [7]:
iso_forest = IsolationForest(n_estimators=200, contamination=0.10, random_state=42, n_jobs=-1)
iso_forest.fit(X_scaled)

iso_pred   = iso_forest.predict(X_scaled)
iso_scores = iso_forest.score_samples(X_scaled)

df_clean = df_clean.copy()
df_clean["iso_anomaly"]    = (iso_pred == -1).astype(int)
df_clean["iso_score"]      = iso_scores
df_clean["iso_score_norm"] = MinMaxScaler().fit_transform((-iso_scores).reshape(-1,1)).flatten()

n_anomalies = df_clean["iso_anomaly"].sum()
print("Anomalies: " + str(n_anomalies) + " (" + str(round(100*n_anomalies/len(df_clean),1)) + "%)")

cross = pd.crosstab(
    df_clean["risk_label"].map({0:"LOW",1:"MEDIUM",2:"HIGH"}),
    df_clean["iso_anomaly"].map({0:"Normal",1:"Anomaly"}),
    rownames=["XGBoost"], colnames=["IsoForest"])
print(cross)


Anomalies: 27 (10.0%)
IsoForest  Anomaly  Normal
XGBoost                   
HIGH             3       0
LOW             13     201
MEDIUM          11      42


## Cell 7 — Combined risk index

In [8]:
xgb_proba = xgb_model.predict_proba(X_scaled)
df_clean["xgb_prob_high"] = xgb_proba[:,2] if xgb_proba.shape[1] > 2 else 0
df_clean["xgb_pred"]      = xgb_model.predict(X_scaled)

xgb_norm = MinMaxScaler().fit_transform(df_clean[["xgb_prob_high"]]).flatten()
df_clean["combined_risk_score"] = 0.6 * xgb_norm + 0.4 * df_clean["iso_score_norm"]

def risk_tier(score):
    if score >= 0.65:   return "HIGH"
    elif score >= 0.35: return "MEDIUM"
    else:               return "LOW"

df_clean["risk_tier"] = df_clean["combined_risk_score"].apply(risk_tier)

tier_counts = df_clean["risk_tier"].value_counts()
print("Final risk tier distribution:")
for tier in ["HIGH","MEDIUM","LOW"]:
    n   = tier_counts.get(tier, 0)
    pct = round(100*n/len(df_clean), 1)
    print("  " + tier + ": " + str(n) + " pixels (" + str(pct) + "%)")

print()
print("Top 10 highest-risk locations:")
top10 = df_clean.nlargest(10, "combined_risk_score")[
    ["lat","lon","combined_risk_score","risk_tier","VV","dark_spot_mask","SAR_change_dB"]].round(3)
print(top10.to_string(index=False))


Final risk tier distribution:
  HIGH: 2 pixels (0.7%)
  MEDIUM: 1 pixels (0.4%)
  LOW: 267 pixels (98.9%)

Top 10 highest-risk locations:
  lat   lon  combined_risk_score risk_tier      VV  dark_spot_mask  SAR_change_dB
5.637 6.625                0.978      HIGH -15.264               1         -0.203
5.727 6.625                0.943      HIGH -11.826               1         -0.315
5.502 6.535                0.493    MEDIUM -15.383               1          1.529
5.682 6.625                0.324       LOW -10.286               0          0.338
5.098 6.805                0.312       LOW  -0.729               0          1.983
5.457 6.535                0.265       LOW  -8.289               0          0.146
5.547 6.535                0.251       LOW  -7.487               0          0.813
5.637 6.850                0.220       LOW  -4.025               0          2.831
5.592 6.580                0.180       LOW  -8.957               0          0.180
5.502 7.074                0.179       LOW

## Cell 8 — Risk score map (saved to outputs/)

In [10]:
import plotly.graph_objects as go

tier_colors = {'HIGH': '#E24B4A', 'MEDIUM': '#EF9F27', 'LOW': '#B5D4F4'}
tier_sizes  = {'HIGH': 12, 'MEDIUM': 8, 'LOW': 4}

fig = go.Figure()

for tier in ['LOW', 'MEDIUM', 'HIGH']:
    mask = df_clean['risk_tier'] == tier
    g    = df_clean[mask]
    fig.add_trace(go.Scatter(
        x=g['lon'], y=g['lat'],
        mode='markers',
        name=tier,
        marker=dict(color=tier_colors[tier],
                    size=tier_sizes[tier], opacity=0.85),
        text=['Score: ' + str(round(s, 3)) + '  VV: ' + str(round(v, 2))
              for s, v in zip(g['combined_risk_score'], g['VV'])],
        hovertemplate='%{text}<extra>' + tier + '</extra>'
    ))

fig.update_layout(
    title='PipelineWatch-NG - Combined Risk Score Map<br>Trans Niger Pipeline corridor',
    xaxis_title='Longitude',
    yaxis_title='Latitude',
    height=520,
    legend=dict(title='Risk tier')
)

fig.write_html(os.path.join(OUTPUT_DIR, 'm3_risk_score_map.html'))
print('Saved: outputs/m3_risk_score_map.html')
from IPython.display import IFrame
display(IFrame(src='../outputs/m3_risk_score_map.html', width='100%', height=550))

Saved: outputs/m3_risk_score_map.html


## Cell 9 — Save models and scored dataset

In [11]:
xgb_model.save_model(os.path.join(MODEL_DIR, "xgb_risk_scorer.json"))
joblib.dump(scaler,     os.path.join(MODEL_DIR, "feature_scaler.pkl"))
joblib.dump(iso_forest, os.path.join(MODEL_DIR, "isolation_forest.pkl"))

scored_path = os.path.join(CACHE_DIR, "m3_risk_scored.csv")
df_clean.to_csv(scored_path, index=False)

config = {"feature_cols": FEATURE_COLS, "model_version": "1.0",
          "trained": datetime.now().isoformat(), "n_samples": len(df_clean),
          "cv_accuracy": round(float(cv_scores.mean()),3),
          "risk_tiers": df_clean["risk_tier"].value_counts().to_dict()}
with open(os.path.join(CACHE_DIR, "m3_model_config.json"), "w") as f:
    json.dump(config, f, indent=2)

print("=" * 40)
print("MODULE 3 COMPLETE")
print("=" * 40)
print("CV accuracy    : " + str(round(cv_scores.mean(),3)))
print("HIGH risk zones: " + str((df_clean["risk_tier"]=="HIGH").sum()))
print("Models in      : data/models/")
print("Plots in       : outputs/")
print("Scored CSV     : data/cached/m3_risk_scored.csv")


MODULE 3 COMPLETE
CV accuracy    : 0.981
HIGH risk zones: 2
Models in      : data/models/
Plots in       : outputs/
Scored CSV     : data/cached/m3_risk_scored.csv


## Module 3 complete

| Output | File |
|--------|------|
| Feature importance chart | outputs/m3_feature_importance.png |
| Risk score map | outputs/m3_risk_score_map.png |
| Risk scored dataset | data/cached/m3_risk_scored.csv |
| XGBoost model | data/models/xgb_risk_scorer.json |
| Isolation Forest | data/models/isolation_forest.pkl |

**Next:** Run Module 2b (TROPOMI dry season validation) or Module 5 (weekly NRT update)